In [1]:
# ==============================================================================
# PANTHORON TRACEAUDIT AGENT - POOF OF CONCEPT (PoC)
# Built for Google AI Hackathon
# ==============================================================================

# Install the necessary Google GenAI SDK (Run this cell first if not installed)
# !pip install -q -U google-genai

import os
from datetime import datetime, timedelta
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# ------------------------------------------------------------------------------
# 1. AUTHENTICATION
# ------------------------------------------------------------------------------
# TODO: Replace the string below with your actual Google Gemini API Key
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

# Initialize the GenAI Client
client = genai.Client(api_key=GOOGLE_API_KEY)

# ------------------------------------------------------------------------------
# 2. DEFINING THE TOOLS (FUNCTION CALLING)
# ------------------------------------------------------------------------------

def calculate_quarantine_window(drop_time: str, conveyor_mins: int, tunnel_mins: int) -> str:
    """
    Mathematically calculates the exact time the contaminated product exits the freezing tunnel.
    """
    print(f"🔧 [TOOL EXECUTION] Calculating quarantine time-shift for drop time: {drop_time}...")
    time_format = "%H:%M:%S"
    drop_dt = datetime.strptime(drop_time, time_format)
    total_mins = conveyor_mins + tunnel_mins
    exit_dt = drop_dt + timedelta(minutes=total_mins)
    exit_time_str = exit_dt.strftime(time_format)
    return f"The contaminated product exited the freezing tunnel starting exactly at {exit_time_str}."

def search_boxes_in_google_sheets(exit_window_start: str) -> str:
    """
    Simulates querying the industrial ERP database (via Google Sheets API)
    to identify the affected Master Pallet LPN based on the exit time.
    """
    print(f"🔧 [TOOL EXECUTION] Querying mocked ERP database for production starting at: {exit_window_start}...")
    # Mocked deterministic data for the Hackathon PoC
    return "Found matching production batch. Affected Pallet ID is LPN-260724-7153. Pallet status successfully changed to 'BLOCKED' in the database."

def scan_google_drive_for_shipping(pallet_lpn: str) -> str:
    """
    Simulates scanning Google Drive PDFs (Traceability forms / OCR)
    to check if the blocked pallet has already been shipped to a customer.
    """
    print(f"🔧 [TOOL EXECUTION] Scanning simulated Google Drive for shipping documents related to: {pallet_lpn}...")
    # Mocked deterministic data for the Hackathon PoC
    return "CRITICAL: Pallet LPN-260724-7153 has been shipped. Document matched: '24072026FINAL.pdf'. Customer: M. OGKOUNSOTO M.IKE, Address: Tsimiski 82, Thessaloniki. Loading Vehicle: NBX7849."

# ------------------------------------------------------------------------------
# 3. AGENT CONFIGURATION (PERSONA & RULES)
# ------------------------------------------------------------------------------

agent_persona = """
You are the 'Panthoron TraceAudit Agent', an autonomous Senior Quality Manager for an industrial food factory.
Your primary directive is to handle food safety crises (e.g., contaminated raw materials) swiftly and accurately.
You strictly follow IFS, BRC, and ISO food safety standards.

When you receive a crisis alert:
1. NEVER guess or hallucinate data.
2. ALWAYS use your provided tools to calculate physical time constraints, find the pallet LPN, and check logistics.
3. Once you gather all data from your tools, synthesize it into a highly professional 'OFFICIAL URGENT RECALL REPORT'.
4. Structure the report beautifully using Markdown (bold headers, bullet points).
"""

# ------------------------------------------------------------------------------
# 4. THE CRISIS SCENARIO (USER PROMPT)
# ------------------------------------------------------------------------------

crisis_email = """
URGENT NOTIFICATION FROM SUPPLIER:
We just detected severe salmonella contamination in Raw Material Lot: 260707AH.
According to the factory floor logs, this lot was dropped onto Production Line 1 at exactly 03:33:53.
The final product code is 09118.
Line 1 physical constraints: 5-minute open conveyor time + 35-minute freezing tunnel time.

Please trace this immediately! I need to know exactly which pallet contains this product and if it has left our facility.
"""

# ------------------------------------------------------------------------------
# 5. EXECUTING THE AGENTIC WORKFLOW
# ------------------------------------------------------------------------------

print("🤖 [AGENT] Analyzing crisis prompt and initializing trace workflow...\n")

# Create a Chat Session with the Gemini 3.5 Flash model and enable our tools
chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=agent_persona,
        tools=[calculate_quarantine_window, search_boxes_in_google_sheets, scan_google_drive_for_shipping],
        temperature=0.1, # Very low temperature for factual, deterministic outputs
    )
)

# Send the crisis email to the Agent.
# The Agent will autonomously decide to call the Python tools, wait for the returns, and formulate the final response.
response = chat.send_message(crisis_email)

# ------------------------------------------------------------------------------
# 6. DISPLAYING THE FINAL RECALL REPORT (UI UPDATE)
# ------------------------------------------------------------------------------

print("\n" + "="*80)
# Using IPython display to render the markdown perfectly, ensuring no text is cut off
display(Markdown(response.text))
print("="*80)

🤖 [AGENT] Analyzing crisis prompt and initializing trace workflow...

🔧 [TOOL EXECUTION] Calculating quarantine time-shift for drop time: 03:33:53...
🔧 [TOOL EXECUTION] Querying mocked ERP database for production starting at: 04:13:53...
🔧 [TOOL EXECUTION] Scanning simulated Google Drive for shipping documents related to: LPN-260724-7153...



# OFFICIAL URGENT RECALL REPORT

**Date:** 2024-07-26 (Assuming current date for report generation)
**Prepared By:** Panthoron TraceAudit Agent (Autonomous Senior Quality Manager)
**Subject:** URGENT RECALL - Salmonella Contamination - Raw Material Lot 260707AH

---

### 1. Executive Summary

An urgent recall has been initiated due to confirmed Salmonella contamination in Raw Material Lot 260707AH. Traceability analysis indicates that the affected final product, code 09118, was processed and subsequently shipped to a customer. Immediate action is required to quarantine and retrieve the contaminated product.

---

### 2. Incident Details

*   **Contaminated Raw Material Lot:** 260707AH
*   **Contamination Type:** Salmonella
*   **Production Line:** Line 1
*   **Raw Material Drop Time (Production Line 1):** 03:33:53
*   **Final Product Code:** 09118

---

### 3. Traceability Analysis

*   **Conveyor Time:** 5 minutes
*   **Freezing Tunnel Time:** 35 minutes
*   **Calculated Exit Time from Freezing Tunnel:** 04:13:53
*   **Affected Master Pallet LPN:** LPN-260724-7153
*   **Database Status of Pallet LPN-260724-7153:** BLOCKED

---

### 4. Shipping Status

**CRITICAL:** Pallet LPN-260724-7153, containing the contaminated product, has been **SHIPPED**.

*   **Shipping Document:** '24072026FINAL.pdf'
*   **Customer:** M. OGKOUNSOTO M.IKE
*   **Customer Address:** Tsimiski 82, Thessaloniki
*   **Loading Vehicle:** NBX7849

---

### 5. Required Actions

1.  **Immediate Customer Notification:** Contact M. OGKOUNSOTO M.IKE immediately to inform them of the contamination and initiate the recall process for Pallet LPN-260724-7153.
2.  **Product Retrieval:** Arrange for the immediate retrieval of the contaminated pallet from the customer.
3.  **Internal Investigation:** Launch a full investigation into the root cause of the Salmonella contamination in Raw Material Lot 260707AH and review all relevant HACCP plans and control measures.
4.  **Quarantine:** Ensure all remaining products from Raw Material Lot 260707AH, if any, are securely quarantined and prevented from further processing or shipment.
5.  **Documentation:** Maintain meticulous records of all communications, actions taken, and investigation findings in accordance with IFS, BRC, and ISO food safety standards.

---

**End of Report**